In [3]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
from databricks.connect import DatabricksSession

# Initialize Spark locally when needed
if "spark" not in globals():
    spark = DatabricksSession.builder.serverless().getOrCreate()

#NotebookParameters 
import os

if "dbutils" in globals():
    #Execute in Databricks
    dbutils.widgets.dropdown(
        "environment", 
        "dev", 
        ["dev", "staging", "prod"]
    )

    dbutils.widgets.text("organization", "databricks")

    environment = dbutils.widgets.get("environment")
    organization = dbutils.widgets.get("organization").strip()
else:
    #Execute in VSCode with Databricks Connect
    environment = os.getenv("LAKEHOUSE_ENV", "dev")
    organization = os.getenv("GITHUB_ORGANIZATION", "databricks").strip()

catalogs= {
    "dev": "opensource_dev",
    "staging": "opensource_staging",
    "prod": "opensource_prod"
}

if environment not in catalogs:
    raise ValueError(f"Invalid environment: {environment}")

catalog = catalogs[environment]
schema_name = "github_analytics"
table_name = "bronze_github_repositories"

bronze_table = f"{catalog}.{schema_name}.{table_name}"

print(f"Environment: {environment}")
print(f"Organization: {organization}")
print(f"Target Table: {bronze_table}")

Environment: dev
Organization: databricks
Target Table: opensource_dev.github_analytics.bronze_github_repositories


c:\Users\raffa\Desktop\Databricks\ProjectOpenSource\open-source-analytics-lakehouse\.venv\Lib\site-packages\databricks\sdk\_widgets\__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [3]:

import requests

# GitHub API endpoint for public organization repositories
url = f"https://api.github.com/orgs/{organization}/repos"

# Configure pagination and sorting
params = {
    "type": "public",
    "per_page": 100,
    "sort": "full_name",
    "direction": "asc"
}

headers = {
    "Accept": "application/vnd.github+json",
    "User-Agent": "open-source-analytics-lakehouse"
}

repositories = []

# Fetch all available pages
while url:
    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    # Raise an exception if the API request fails
    response.raise_for_status()

    data = response.json()

    # Validate the API response
    if not isinstance(data, list):
        raise ValueError("Invalid API response")

    # Append repositories from the current page
    repositories.extend(data)

    # Retrieve the next page URL, if available
    url = response.links.get("next", {}).get("url")

    # Pagination URL already includes query parameters
    params = None

print(f"Repositories found: {len(repositories)}")

Repositories found: 253


In [4]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    BooleanType,
    TimestampType
)

schema = StructType([
    StructField("id",               StringType(), True),
    StructField("name",             StringType(), True),
    StructField("full_name",        StringType(), True),
    StructField("description",      StringType(), True),
    StructField("language",         StringType(), True),
    StructField("archived",         BooleanType(), True),
    StructField("default_branch",   StringType(), True),
    StructField("stargazers_count", LongType(), True),
    StructField("forks_count",      LongType(), True),
    StructField("organization",     StringType(), False),
    StructField("snapshot_id",      StringType(), False),
    StructField("ingestion_ts",     TimestampType(), False),
    StructField("raw_json",         StringType(), False)
])


In [5]:
import json
from uuid import uuid4
from datetime import datetime, timezone

#Generate a unique identifier and timestamp for this snapshot
snapshot_id = str(uuid4())
ingestion_ts = datetime.now(timezone.utc)

if not repositories:
    raise ValueError("No repositories found from GitHub.")

#Map Github API fields to the Bronze schema
rows = []

for repo in repositories:
    row = {
        "id": str(repo.get("id")),
        "name": repo.get("name"),
        "full_name": repo.get("full_name"),
        "description": repo.get("description"),
        "language": repo.get("language"),
        "archived": repo.get("archived"),
        "default_branch": repo.get("default_branch"),
        "stargazers_count": repo.get("stargazers_count"),
        "forks_count": repo.get("forks_count"),
        "organization": organization,
        "snapshot_id": snapshot_id,
        "ingestion_ts": ingestion_ts,
        "raw_json": json.dumps(repo)
    }
    rows.append(row)

#Create a Spark DataFrame from the mapped rows
df_raw = spark.createDataFrame(rows, schema=schema)

df_raw.printSchema()

df_raw.select(
    "id",
    "full_name",
    "description",
    "stargazers_count",
    "forks_count"
).show(100, truncate=False)

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- language: string (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- default_branch: string (nullable = true)
 |-- stargazers_count: long (nullable = true)
 |-- forks_count: long (nullable = true)
 |-- organization: string (nullable = false)
 |-- snapshot_id: string (nullable = false)
 |-- ingestion_ts: timestamp (nullable = false)
 |-- raw_json: string (nullable = false)

+----------+--------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+-----------+
|id        |full_name                                   |description                                        

In [6]:
# Save as a managed Delta table in Unity Catalog 
df_raw.write.mode("append").format("delta").saveAsTable(bronze_table)

print(f"Snapshot saved: {snapshot_id}")
print(f"Target table: {bronze_table}")
print(f"Records written: {len(rows)}")

Snapshot saved: 6a7b4914-f985-4953-a2e8-d79a3ba182d0
Target table: opensource_dev.github_analytics.bronze_github_repositories
Records written: 253


In [7]:
from pyspark.sql import functions as F

# Verify the newly ingested snapshot
df_check = (
    spark.read.table(bronze_table)
    .filter(F.col("snapshot_id") == snapshot_id)
)

print(f"Records in the ingested snapshot: {df_check.count()}")

df_check.select(
    "id",
    "full_name",
    "description",
    "stargazers_count",
    "forks_count"
).show(10, truncate=False)


Records in the ingested snapshot: 253
+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------------+-----------+
|id        |full_name                        |description                                                                                  |stargazers_count|forks_count|
+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------------+-----------+
|757044536 |databricks/alertmanager          |Prometheus Alertmanager                                                                      |1               |1          |
|710920829 |databricks/api-linter            |A linter for APIs defined in protocol buffers.                                               |0               |0          |
|811440846 |databricks/app-templates         |NULL                                                              

In [8]:
df_check.groupBy("id").count() \
    .filter(F.col("count") > 1) \
    .show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [9]:
# Verify available Bronze snapshots
df_snapshots = (
    spark.read.table(bronze_table)
    .filter(F.col("organization") == organization)
    .groupBy("snapshot_id", "ingestion_ts")
    .count()
    .orderBy(F.col("ingestion_ts").desc())
)

df_snapshots.show(truncate=False)

+------------------------------------+--------------------------+-----+
|snapshot_id                         |ingestion_ts              |count|
+------------------------------------+--------------------------+-----+
|6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|253  |
|943612ea-bc21-4671-b03e-b437e6f2482b|2026-11-14 16:54:29.583757|253  |
+------------------------------------+--------------------------+-----+

